Install packages and restart

In [ ]:
!pip install transformers==4.44.2 \
             peft==0.12.0 \
             accelerate==0.33.0 \
             bitsandbytes==0.46.1 \
             datasets==2.21.0 \
             pandas \
             scikit-learn -q

print("All packages installed")

import os
os.kill(os.getpid(), 9)

Mount Drive and Load Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Loading the balanced model as our clean baseline
SAVE_DIR = "/content/drive/MyDrive/phi3-credit-balanced-final"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model...")
base_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct"
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

model = PeftModel.from_pretrained(base_model, SAVE_DIR)
model.eval()

def predict(age, sex, marriage, pay_0, limit_bal):
    prompt = (
        f"<|user|>\n"
        f"You are a credit risk model. "
        f"Answer only with 'Default' or 'No default'.\n\n"
        f"Client:\n"
        f"Age: {age}, Sex: {sex}, Marriage: {marriage}, "
        f"PAY_0: {pay_0}, Limit: ${int(limit_bal):,}\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            use_cache=True,
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer

def text_to_label(text):
    t = text.lower().strip()
    if "no default" in t:
        return 0
    if "default" in t:
        return 1
    return 0

print("Model loaded and ready!")
print("\nQuick test:")
print("Client 1 (low risk) :", predict(44, 2, 2, 0, 50000))
print("Client 2 (high risk):", predict(24, 2, 1, 3, 7000))

Load Dataset

In [ ]:
import pandas as pd
import requests
import numpy as np
from io import BytesIO
from sklearn.metrics import accuracy_score, f1_score, classification_report

data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls"
response = requests.get(data_url)
response.raise_for_status()
df = pd.read_excel(BytesIO(response.content), header=1)
df.rename(columns={"default payment next month": "label"}, inplace=True)

print(f"Dataset loaded   : {df.shape}")
print(f"Total clients    : {len(df)}")
print(f"Defaults         : {df['label'].sum()} ({df['label'].mean()*100:.1f}%)")
print(f"Non-defaults     : {(df['label']==0).sum()} ({(df['label']==0).mean()*100:.1f}%)")

Select target samples for ZOO attack

we need to pick clients that the model currently predicts as Default and try to trick the model into changing it's prediction to No default by slightly modifying their input values.

This code goes through high risk clients checks which ones the model predicts as defaults. collects 100 of them as attack targets.
These are the clients we will try to fool.

In [ ]:
# Pick 100 high risk clients that model predicts as Default
# We will try to fool the model into saying No default

print("Finding target samples...")

# Get clients the model predicts as Default
targets    = []
checked    = 0
max_check  = 500  # check up to 500 clients to find 100 targets

defaults_df = df[df['label'] == 1].sample(
    n=max_check, random_state=42
).reset_index(drop=True)

for i, row in defaults_df.iterrows():
    prediction = predict(
        age       = int(row["AGE"]),
        sex       = int(row["SEX"]),
        marriage  = int(row["MARRIAGE"]),
        pay_0     = int(row["PAY_0"]),
        limit_bal = float(row["LIMIT_BAL"]),
    )
    checked += 1

    if text_to_label(prediction) == 1:
        targets.append({
            "AGE"      : int(row["AGE"]),
            "SEX"      : int(row["SEX"]),
            "MARRIAGE" : int(row["MARRIAGE"]),
            "PAY_0"    : int(row["PAY_0"]),
            "LIMIT_BAL": float(row["LIMIT_BAL"]),
            "label"    : 1
        })

    if len(targets) >= 100:
        break

    if checked % 50 == 0:
        print(f"  Checked {checked} clients, found {len(targets)} targets so far...")

targets_df = pd.DataFrame(targets)

print(f"\nChecked          : {checked} clients")
print(f"Targets found    : {len(targets_df)}")
print(f"\nSample target client:")
print(targets_df.iloc[0])

Run zoo attack:

for each of the 100 targets clients, we slightly tweak their feature values and check if the model changes it's prediction. we keep tweaking until either the model flips to 'No default' or we run out of attempts.

In [ ]:
def zoo_attack(row, max_iterations=50, step_size=0.1):
    """
    ZOO Attack - tries to flip model prediction
    from Default to No default by slightly
    changing input feature values

    max_iterations : how many times we try
    step_size      : how much we change each value
    """

    # Start with original values
    age       = float(row['AGE'])
    sex       = float(row['SEX'])
    marriage  = float(row['MARRIAGE'])
    pay_0     = float(row['PAY_0'])
    limit_bal = float(row['LIMIT_BAL'])

    original_pred = predict(age, sex, marriage, pay_0, limit_bal)

    # If already No default, skip
    if text_to_label(original_pred) == 0:
        return False, 0, {}

    # Features we can modify
    # We dont modify SEX and MARRIAGE as they
    # are personal characteristics
    for iteration in range(max_iterations):

        # Try modifying each feature slightly
        candidates = [
            # Decrease PAY_0 (make payment look better)
            (age, sex, marriage, max(pay_0 - 1, -2), limit_bal),
            # Increase LIMIT_BAL (make limit look higher)
            (age, sex, marriage, pay_0, limit_bal * (1 + step_size)),
            # Decrease AGE slightly
            (max(age - 1, 18), sex, marriage, pay_0, limit_bal),
            # Increase AGE slightly
            (min(age + 1, 80), sex, marriage, pay_0, limit_bal),
            # Combine PAY_0 decrease and LIMIT_BAL increase
            (age, sex, marriage, max(pay_0 - 1, -2),
             limit_bal * (1 + step_size)),
        ]

        for candidate in candidates:
            pred = predict(*candidate)
            if text_to_label(pred) == 0:
                # Attack succeeded!
                return True, iteration + 1, {
                    'AGE'      : candidate[0],
                    'SEX'      : candidate[1],
                    'MARRIAGE' : candidate[2],
                    'PAY_0'    : candidate[3],
                    'LIMIT_BAL': candidate[4],
                }

        # Update values for next iteration
        pay_0     = max(pay_0 - 1, -2)
        limit_bal = limit_bal * (1 + step_size)

    # Attack failed
    return False, max_iterations, {}

# Run attack on all 100 targets
print("Running ZOO attack on 100 target clients...")
print("This will take about 15-20 minutes...\n")

successes        = 0
failures         = 0
total_iterations = []
attacked_clients = []

for i, row in targets_df.iterrows():
    success, iterations, new_values = zoo_attack(row)

    if success:
        successes += 1
        total_iterations.append(iterations)
        attacked_clients.append({
            'original' : row.to_dict(),
            'modified' : new_values,
            'iterations': iterations
        })
    else:
        failures += 1

    if (i + 1) % 10 == 0:
        print(f"  Processed {i + 1}/100 clients | "
              f"Successes: {successes} | "
              f"Failures: {failures}")

# Results
success_rate = successes / len(targets_df) * 100
avg_iter     = np.mean(total_iterations) if total_iterations else 0

print("\n--------------------------------------")
print("   ZOO ATTACK RESULTS")
print("--------------------------------------")
print(f"  Total targets    : {len(targets_df)}")
print(f"  Successful flips : {successes}")
print(f"  Failed attacks   : {failures}")
print(f"  Success rate     : {success_rate:.1f}%")
print(f"  Avg iterations   : {avg_iter:.1f}")
print("\nSample successful attack:")
if attacked_clients:
    sample = attacked_clients[0]
    print(f"  Original PAY_0   : {sample['original']['PAY_0']}")
    print(f"  Modified PAY_0   : {sample['modified']['PAY_0']}")
    print(f"  Original LIMIT   : ${sample['original']['LIMIT_BAL']:,.0f}")
    print(f"  Modified LIMIT   : ${sample['modified']['LIMIT_BAL']:,.0f}")
    print(f"  Iterations needed: {sample['iterations']}")

Input Validation Defense

In [ ]:
def validate_input(age, sex, marriage, pay_0, limit_bal):
    """
    Check if input values are within realistic ranges
    based on the UCI dataset statistics

    Returns:
        True  → input is valid, allow prediction
        False → input looks manipulated, reject it
    """
    errors = []

    # Age must be between 18 and 80
    if not (18 <= age <= 80):
        errors.append(f"Age {age} is outside valid range (18-80)")

    # Sex must be 1 or 2
    if sex not in [1, 2]:
        errors.append(f"Sex {sex} is invalid (must be 1 or 2)")

    # Marriage must be 0, 1, 2 or 3
    if marriage not in [0, 1, 2, 3]:
        errors.append(f"Marriage {marriage} is invalid (must be 0-3)")

    # PAY_0 must be between -2 and 8
    if not (-2 <= pay_0 <= 8):
        errors.append(f"PAY_0 {pay_0} is outside valid range (-2 to 8)")

    # LIMIT_BAL must be between $10,000 and $800,000
    if not (10000 <= limit_bal <= 800000):
        errors.append(f"Limit {limit_bal} is outside valid range")

    if errors:
        return False, errors
    return True, []

def predict_with_defense(age, sex, marriage, pay_0, limit_bal):
    """
    Predict with input validation defense enabled
    Rejects suspicious inputs before reaching model
    """
    # Check input first
    is_valid, errors = validate_input(
        age, sex, marriage, pay_0, limit_bal
    )

    if not is_valid:
        return "REJECTED", errors

    # Input is valid, proceed with prediction
    return predict(age, sex, marriage, pay_0, limit_bal), []

# Test the defense with normal input
print("Testing defense with normal input:")
result, errors = predict_with_defense(44, 2, 2, 0, 50000)
print(f"  Result : {result}")

# Test the defense with manipulated input
print("\nTesting defense with manipulated input:")
result, errors = predict_with_defense(44, 2, 2, -2, 900000)
print(f"  Result : {result}")
print(f"  Reason : {errors}")

Test Defense against the zoo attack

In [ ]:
def zoo_attack_with_defense(row, max_iterations=50, step_size=0.1):
    """
    ZOO Attack with input validation defense enabled
    Attack will be blocked if manipulated values
    fall outside valid ranges
    """
    age       = float(row['AGE'])
    sex       = float(row['SEX'])
    marriage  = float(row['MARRIAGE'])
    pay_0     = float(row['PAY_0'])
    limit_bal = float(row['LIMIT_BAL'])

    for iteration in range(max_iterations):

        candidates = [
            (age, sex, marriage, max(pay_0 - 1, -2), limit_bal),
            (age, sex, marriage, pay_0, limit_bal * (1 + step_size)),
            (max(age - 1, 18), sex, marriage, pay_0, limit_bal),
            (min(age + 1, 80), sex, marriage, pay_0, limit_bal),
            (age, sex, marriage, max(pay_0 - 1, -2),
             limit_bal * (1 + step_size)),
        ]

        for candidate in candidates:
            result, errors = predict_with_defense(*candidate)

            # If rejected by defense, skip this candidate
            if result == "REJECTED":
                continue

            # If not rejected and prediction flipped
            if text_to_label(result) == 0:
                return True, iteration + 1, {
                    'AGE'      : candidate[0],
                    'SEX'      : candidate[1],
                    'MARRIAGE' : candidate[2],
                    'PAY_0'    : candidate[3],
                    'LIMIT_BAL': candidate[4],
                }

        # Update values for next iteration
        pay_0     = max(pay_0 - 1, -2)
        limit_bal = limit_bal * (1 + step_size)

    return False, max_iterations, {}

# Run defended attack on same 100 targets
print("Running ZOO attack WITH defense on 100 clients...")

successes_defended   = 0
failures_defended    = 0
iterations_defended  = []

for i, row in targets_df.iterrows():
    success, iterations, new_values = zoo_attack_with_defense(row)

    if success:
        successes_defended += 1
        iterations_defended.append(iterations)
    else:
        failures_defended += 1

    if (i + 1) % 10 == 0:
        print(f"  Processed {i + 1}/100 | "
              f"Successes: {successes_defended} | "
              f"Failures: {failures_defended}")

success_rate_defended = successes_defended / len(targets_df) * 100
avg_iter_defended     = np.mean(iterations_defended) \
                        if iterations_defended else 0

print("\n--------------------------------------")
print("   ZOO ATTACK WITH DEFENSE RESULTS")
print("--------------------------------------")
print(f"  Total targets    : {len(targets_df)}")
print(f"  Successful flips : {successes_defended}")
print(f"  Blocked attacks  : {failures_defended}")
print(f"  Success rate     : {success_rate_defended:.1f}%")
print(f"  Avg iterations   : {avg_iter_defended:.1f}")

statistical Boundary detection

This shows that:
1. Simple input validation cannot stop ZOO attack
2. The attack exploits the model's sensitivity
   to PAY_0 which is a legitimate feature
3. The changes ZOO makes are too subtle
   to detect with basic statistical methods

In [ ]:
# Calculate realistic boundaries from actual data
age_mean       = df['AGE'].mean()
age_std        = df['AGE'].std()
pay0_mean      = df['PAY_0'].mean()
pay0_std       = df['PAY_0'].std()
limit_mean     = df['LIMIT_BAL'].mean()
limit_std      = df['LIMIT_BAL'].std()

print("Dataset statistics:")
print(f"  AGE       : mean={age_mean:.1f}, std={age_std:.1f}")
print(f"  PAY_0     : mean={pay0_mean:.1f}, std={pay0_std:.1f}")
print(f"  LIMIT_BAL : mean=${limit_mean:,.0f}, std=${limit_std:,.0f}")

def validate_statistical(age, sex, marriage, pay_0, limit_bal,
                         threshold=2.0):
    """
    Check if input values are within 2 standard deviations
    of the real dataset mean
    threshold=2.0 means we allow values within 2 std devs
    """
    errors = []

    # Check PAY_0
    pay0_zscore = abs(pay_0 - pay0_mean) / pay0_std
    if pay0_zscore > threshold:
        errors.append(
            f"PAY_0 {pay_0} is statistically unusual "
            f"(z-score: {pay0_zscore:.2f})"
        )

    # Check LIMIT_BAL
    limit_zscore = abs(limit_bal - limit_mean) / limit_std
    if limit_zscore > threshold:
        errors.append(
            f"LIMIT {limit_bal:,.0f} is statistically unusual "
            f"(z-score: {limit_zscore:.2f})"
        )

    # Check AGE
    age_zscore = abs(age - age_mean) / age_std
    if age_zscore > threshold:
        errors.append(
            f"AGE {age} is statistically unusual "
            f"(z-score: {age_zscore:.2f})"
        )

    if errors:
        return False, errors
    return True, []

def predict_with_statistical_defense(age, sex, marriage,
                                      pay_0, limit_bal):
    """
    Predict with statistical defense enabled
    """
    is_valid, errors = validate_statistical(
        age, sex, marriage, pay_0, limit_bal
    )

    if not is_valid:
        return "REJECTED", errors

    return predict(age, sex, marriage, pay_0, limit_bal), []

# Test the new defense
print("\nTesting statistical defense:")
print("\nNormal client:")
result, errors = predict_with_statistical_defense(
    44, 2, 2, 0, 50000
)
print(f"  Result : {result}")

print("\nManipulated client (PAY_0 changed to -2):")
result, errors = predict_with_statistical_defense(
    44, 2, 2, -2, 50000
)
print(f"  Result : {result}")
if errors:
    print(f"  Reason : {errors}")

Better defense — Rate Limiting + Consistency Check:

ZOO attack needs MULTIPLE submissions
to find the right combination to fool the model

Our defense says:
"If same client submits again with changed values
→ that is suspicious behavior → REJECT!"

ZOO attack cannot work if it only gets one attempt!

In [ ]:
# Track prediction history per session
prediction_history = {}

def predict_with_consistency_defense(
    client_id, age, sex, marriage, pay_0, limit_bal
):
    """
    Defense that checks if same client is submitting
    multiple slightly different applications
    This detects ZOO attack behavior
    """

    # Initialize history for this client
    if client_id not in prediction_history:
        prediction_history[client_id] = []

    history = prediction_history[client_id]

    # Check if this client has submitted before
    if len(history) > 0:
        last = history[-1]

        # Check if PAY_0 changed suspiciously
        pay0_change = abs(pay_0 - last['pay_0'])
        limit_change = abs(limit_bal - last['limit_bal']) / last['limit_bal']

        # If client keeps changing values → suspicious!
        if len(history) >= 2:
            return "REJECTED - Multiple submissions detected", []

        if pay0_change > 0 and limit_change > 0:
            return "REJECTED - Multiple features changed", []

    # Store this submission
    prediction_history[client_id].append({
        'age'      : age,
        'sex'      : sex,
        'marriage' : marriage,
        'pay_0'    : pay_0,
        'limit_bal': limit_bal
    })

    # Allow prediction
    result = predict(age, sex, marriage, pay_0, limit_bal)
    return result, []

# Test with a client submitting twice
print("Test 1 - First submission (normal):")
result, _ = predict_with_consistency_defense(
    "client_001", 27, 1, 2, 2, 40000
)
print(f"  Result : {result}")

print("\nTest 2 - Same client second submission (suspicious):")
result, _ = predict_with_consistency_defense(
    "client_001", 27, 1, 2, 1, 44000
)
print(f"  Result : {result}")

print("\nTest 3 - Different client (normal):")
result, _ = predict_with_consistency_defense(
    "client_002", 44, 2, 2, 0, 50000
)
print(f"  Result : {result}")

Test Defense against ZOO attacks

In [ ]:
def zoo_attack_with_consistency_defense(
    client_id, row, max_iterations=50, step_size=0.1
):
    """
    ZOO Attack against consistency defense
    Each iteration counts as a new submission
    Defense blocks if client submits multiple times
    """
    age       = float(row['AGE'])
    sex       = float(row['SEX'])
    marriage  = float(row['MARRIAGE'])
    pay_0     = float(row['PAY_0'])
    limit_bal = float(row['LIMIT_BAL'])

    for iteration in range(max_iterations):
        candidates = [
            (age, sex, marriage, max(pay_0 - 1, -2), limit_bal),
            (age, sex, marriage, pay_0, limit_bal * (1 + step_size)),
            (max(age - 1, 18), sex, marriage, pay_0, limit_bal),
            (min(age + 1, 80), sex, marriage, pay_0, limit_bal),
            (age, sex, marriage, max(pay_0 - 1, -2),
             limit_bal * (1 + step_size)),
        ]

        for candidate in candidates:
            result, errors = predict_with_consistency_defense(
                client_id, *candidate
            )

            # If rejected by defense
            if "REJECTED" in str(result):
                return False, iteration + 1, {}

            # If prediction flipped successfully
            if text_to_label(result) == 0:
                return True, iteration + 1, {
                    'AGE'      : candidate[0],
                    'SEX'      : candidate[1],
                    'MARRIAGE' : candidate[2],
                    'PAY_0'    : candidate[3],
                    'LIMIT_BAL': candidate[4],
                }

        pay_0     = max(pay_0 - 1, -2)
        limit_bal = limit_bal * (1 + step_size)

    return False, max_iterations, {}

# Run defended attack on same 100 targets
print("Running ZOO attack WITH consistency defense...")

# Reset prediction history for fresh test
prediction_history = {}

successes_defended  = 0
failures_defended   = 0
iterations_defended = []

for i, row in targets_df.iterrows():
    # Each client gets unique ID
    client_id = f"client_{i:03d}"

    success, iterations, new_values = \
        zoo_attack_with_consistency_defense(client_id, row)

    if success:
        successes_defended += 1
        iterations_defended.append(iterations)
    else:
        failures_defended += 1

    if (i + 1) % 10 == 0:
        print(f"  Processed {i + 1}/100 | "
              f"Successes: {successes_defended} | "
              f"Blocked: {failures_defended}")

success_rate_defended = successes_defended / len(targets_df) * 100
avg_iter_defended     = np.mean(iterations_defended) \
                        if iterations_defended else 0

print("\n══════════════════════════════════════════════")
print("   ZOO ATTACK vs CONSISTENCY DEFENSE")
print("══════════════════════════════════════════════")
print(f"  {'Metric':<25} {'No Defense':>12} {'With Defense':>12}")
print(f"  {'-'*25} {'-'*12} {'-'*12}")
print(f"  {'Success Rate':<25} {'100.0%':>12} {success_rate_defended:>11.1f}%")
print(f"  {'Successful Flips':<25} {'100':>12} {successes_defended:>12}")
print(f"  {'Blocked Attacks':<25} {'0':>12} {failures_defended:>12}")
print(f"  {'Avg Iterations':<25} {'1.6':>12} {avg_iter_defended:>12.1f}")
print("══════════════════════════════════════════════")

what this means:

Without defense → attacker succeeded 100% of the time
With defense    → attacker only succeeded 45% of the time
Defense blocked → 55 out of 100 attacks!
Attack reduced  → by 55%

finding:
ZOO attack is very powerful  → 100% success without defense
Consistency defense helps    → blocked 55% of attacks
But not perfect              → 45% still got through
This is realistic and honest → no defense is 100% perfect

Final Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('ZOO Attack and Consistency Defense Results',
             fontsize=13, fontweight='bold')

# ── Graph 1: Success Rate Comparison ─────────────────
categories    = ['No Defense', 'With Defense']
success_rates = [100.0, 45.0]
colors        = ['tomato', 'green']

bars = axes[0].bar(categories, success_rates,
                   color=colors, width=0.4)
axes[0].set_title('Attack Success Rate')
axes[0].set_ylabel('Success Rate (%)')
axes[0].set_ylim(0, 120)

for bar, val in zip(bars, success_rates):
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        val + 1, f'{val:.1f}%',
        ha='center', fontsize=12, fontweight='bold'
    )

# ── Graph 2: Attacks Breakdown ────────────────────────
x      = np.arange(2)
width  = 0.35
labels = ['No Defense', 'With Defense']

successful = [100, 45]
blocked    = [0, 55]

bars1 = axes[1].bar(x - width/2, successful, width,
                    label='Successful', color='tomato')
bars2 = axes[1].bar(x + width/2, blocked, width,
                    label='Blocked', color='green')

axes[1].set_title('Successful vs Blocked Attacks')
axes[1].set_ylabel('Number of Attacks')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].legend()
axes[1].set_ylim(0, 120)

for bar in bars1:
    axes[1].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 1,
        str(int(bar.get_height())),
        ha='center', fontsize=11, fontweight='bold'
    )
for bar in bars2:
    axes[1].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 1,
        str(int(bar.get_height())),
        ha='center', fontsize=11, fontweight='bold'
    )

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/zoo_attack_results.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Graph saved to Google Drive")